# 09 — Figures, Paper Evidence Report, and Quality Assurance

Genera figuras vectoriales, reportes listos para revisión y una auditoría final de todos los módulos.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar resultados

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

import matplotlib.pyplot as plt
from reporting import horizontal_bar, line_chart

def optional_csv(name):
    path = OUTPUT_DIR / name
    return pd.read_csv(path, encoding="utf-8-sig") if path.exists() else pd.DataFrame()

annual = optional_csv("05_documents_by_year.csv")
sources = optional_csv("06_source_statistics.csv")
authors = optional_csv("06_author_statistics.csv")
institutions = optional_csv("06_institution_statistics.csv")
countries = optional_csv("06_country_statistics.csv")
keywords = optional_csv("03_author_keyword_frequency.csv")
themes = optional_csv("04_theme_statistics.csv")
theme_year = optional_csv("05_theme_counts_by_year.csv")
related = optional_csv("07_related_work_candidates.csv")


## 2. Figuras bibliométricas

In [ ]:

formats = CONFIG.get("figure_formats", ["png", "pdf", "svg"])
dpi = int(CONFIG.get("figure_dpi", 600))
complete_annual = annual[annual["Year status"] == "complete"] if "Year status" in annual.columns else annual
line_chart(complete_annual, "Year", "Documents", "Annual scientific production", FIGURES_DIR / "annual_scientific_production", formats, dpi)
horizontal_bar(sources, "Source title normalized", "Documents", "Top publication sources", FIGURES_DIR / "top_publication_sources", formats, dpi, 15)
horizontal_bar(countries, "Country", "Documents", "Top contributing countries", FIGURES_DIR / "top_contributing_countries", formats, dpi, 15)
horizontal_bar(keywords, "Keyword normalized", "Documents", "Top author keywords", FIGURES_DIR / "top_author_keywords", formats, dpi, 20)
for family, filename in [("Technology", "top_technologies"), ("Task", "top_tasks"), ("Semantic/Assistance", "semantic_assistance_indicators")]:
    subset = themes[themes["Family"] == family].sort_values("Documents", ascending=False)
    horizontal_bar(subset, "Concept", "Documents", family + " indicators", FIGURES_DIR / filename, formats, dpi, 15)
print("Figures created in:", FIGURES_DIR)


## 3. Tendencias de tecnologías principales

In [ ]:

if not theme_year.empty:
    technology_year = theme_year[theme_year["Family"] == "Technology"].copy()
    totals = technology_year.groupby("Concept")["Documents"].sum().sort_values(ascending=False)
    top_concepts = list(totals.head(6).index)
    trend = technology_year[technology_year["Concept"].isin(top_concepts)].copy()
    excluded = set(CONFIG.get("partial_or_future_years", []))
    trend = trend[~trend["Year"].isin(excluded)]
    line_chart(trend, "Year", "Documents", "Evolution of leading technologies", FIGURES_DIR / "technology_trends", formats, dpi, group="Concept")
    save_csv(trend, OUTPUT_DIR / "09_technology_trends_plotted.csv")


## 4. Reporte de evidencia para el paper

In [ ]:

report = []
report.append("# Bibliometric Evidence Report")
report.append("")
report.append("## Search metadata")
report.append("- Database: " + CONFIG.get("search_database", ""))
report.append("- Search date: " + CONFIG.get("search_date", ""))
report.append("- Corpus: {} documents".format(int(annual["Documents"].sum()) if not annual.empty else "unknown"))
report.append("")
report.append("## Leading sources")
report.append(markdown_table(sources, 20))
report.append("")
report.append("## Leading countries")
report.append(markdown_table(countries, 20))
report.append("")
report.append("## Leading author keywords")
report.append(markdown_table(keywords, 30))
report.append("")
report.append("## Domain structure")
report.append(markdown_table(themes, 200))
report.append("")
report.append("## Research gap caution")
report.append("Low or zero frequency inside this corpus is exploratory evidence. It must be triangulated with targeted searches and manual reading before being stated as a scientific gap.")
report.append("")
report.append("## Related Work candidates")
report.append(markdown_table(related[[c for c in ["Related Work section", "Title", "Year", "Source title normalized", "Cited by", "Selection score"] if c in related.columns]], 120))
(REPORTS_DIR / "bibliometric_evidence_report.md").write_text("\n\n".join(report), encoding="utf-8")


## 5. Textos y captions sugeridos

In [ ]:

captions = """
# Suggested figure captions

1. Annual scientific production related to vision-based perception in agricultural robotics. The current partial year should be identified explicitly or excluded from growth comparisons.
2. Keyword co-occurrence network derived from Author Keywords using fractional counting and the audited thesaurus.
3. Overlay visualization of Author Keywords according to average publication year.
4. Leading countries contributing to vision-based perception research in agricultural robotics.
5. Distribution of dominant technologies and robotic tasks identified in the corpus.
""".strip()
(REPORTS_DIR / "suggested_figure_captions.md").write_text(captions, encoding="utf-8")

integration = """
# Paper integration guide

## State-of-the-art review
Use annual production, keyword networks, leading countries, sources, and thematic statistics to describe the global structure of the field.

## Related Work
Use outputs/07_related_work_candidates.csv as a discovery list, then read and manually select the final references for the three manuscript subsections.

## Research Gap
Contrast dominant tasks such as harvesting, navigation, detection, segmentation, and manipulation against the frequency and content of semantic perception, scene understanding, activity/task recognition, human-object interaction, and assistance prediction. Do not infer novelty from frequency alone.
""".strip()
(REPORTS_DIR / "paper_integration_guide.md").write_text(integration, encoding="utf-8")


## 6. Quality assurance final

In [ ]:

required_outputs = [
    "00_project_summary.csv", "01_corpus_validation.json", "corpus_cleaned.csv",
    "03_author_keyword_frequency.csv", "04_theme_statistics.csv", "05_documents_by_year.csv",
    "06_source_statistics.csv", "06_author_statistics.csv", "06_country_statistics.csv",
    "07_related_work_candidates.csv", "08_ris_status.csv"
]
checks = []
for name in required_outputs:
    path = OUTPUT_DIR / name
    checks.append({"Output": name, "Exists": path.exists(), "Size bytes": path.stat().st_size if path.exists() else 0, "Status": "PASS" if path.exists() and path.stat().st_size > 0 else "FAIL"})
qa = pd.DataFrame(checks)
save_csv(qa, VALIDATION_DIR / "final_output_quality_assurance.csv")

manifest_rows = []
for folder in [OUTPUT_DIR, FIGURES_DIR, REPORTS_DIR, THESAURUS_DIR]:
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            manifest_rows.append({"File": str(path.relative_to(PROJECT_ROOT)), "Size bytes": path.stat().st_size})
manifest = pd.DataFrame(manifest_rows)
save_csv(manifest, VALIDATION_DIR / "output_manifest.csv")
display(qa)
if (qa["Status"] == "FAIL").any():
    raise RuntimeError("Final QA failed. Review validation/final_output_quality_assurance.csv")
print("Final QA: PASS")
